In [1]:
# %matplotlib inline
# %matplotlib notebook
%matplotlib qt5

from IPython.display import Latex
from IPython.display import SVG

import re
import os
import sh
from functools import reduce
import operator as opr

import numpy as np
import pandas as pd
import matplotlib.pyplot   as mplt
import matplotlib.gridspec as gridspec
import matplotlib.cm       as cm
from mpl_toolkits.axes_grid1 import make_axes_locatable

mplt.rcParams.update(
    {'font.size': 20, 'lines.linewidth':2})

import pyaccel
from pymodels import si as si_lat

import pycolleff.impedances as imp
from pycolleff.impedances import element_and_budget as elbud_mod
import pycolleff.rings.sirius as si
import pycolleff.colleff as colleff

# Define some functions

In [61]:
def plota_budget_Zt(bud, pl='Zdy', limsx=(1e-4, 0.1, 150), limsy=(1e2, 1000, 1e4)):
    bud2 = bud.copy()
    fig = mplt.figure(figsize=(12, 10))

    gs = mplt.GridSpec(2, 1)
    gs.update(left=0.11, right=0.98, top=0.98, bottom=0.22, hspace=0.08)

    ax1 = mplt.subplot(gs[0,0])

    ax2 = mplt.subplot(gs[1,0],sharex=ax1)

    if len(limsx) == 3:
        ax1.set_xscale('symlog', linthresh=limsx[1])
        ax2.set_xscale('symlog', linthresh=limsx[1])

    if len(limsy) == 3:
        ax1.set_yscale('symlog', linthresh=limsy[1])
        ax2.set_yscale('symlog', linthresh=limsy[1])

    ax1.grid(True)
    ax2.grid(True)

    N = len(bud)
    for i in range(len(bud2)):
        Z = getattr(bud2[-1],pl)
        if Z is None or len(Z)>0:
            ind = bud2.ang_freq >= 0
            freq = bud2.ang_freq[ind]/2/np.pi * 1e-9
            Z = getattr(bud2, pl)[ind]*elbud_mod._FACTOR[pl] * 1e-3
            lab  = bud2[-1].name
            cor = i/N if i % 2 else (N-i)/N
            ax1.fill_between(freq, Z.real, color=mplt.cm.jet(cor), label=lab)
            ax2.fill_between(freq, Z.imag, color=mplt.cm.jet(cor), label=lab)
            
            lw = 0.5
            lc = [1, 1, 1]
            lc = [0, 0, 0]
            ax1.plot(freq, Z.real, color=lc, linewidth=lw)
            ax2.plot(freq, Z.imag, color=lc, linewidth=lw)
        bud2.pop()

    lab = imp.Budget._YLABEL['Zdy'].replace('k', 'M').replace('es ', 'es Re ')
    ax1.set_ylabel(lab)
    lab = imp.Budget._YLABEL['Zdy'].replace('k', 'M').replace('es ', 'es Im ')
    ax2.set_ylabel(lab)
    ax2.set_xlabel(r'Frequency [GHz]')
    
    mplt.setp(ax1.get_xticklabels(), visible=False)
    
    mix, *_, mx = limsx
    ax1.set_xlim([mix, mx])

    miy, *_, may = limsy
    ax1.set_ylim([miy, may])

    ax2.set_ylim([-may, -miy])

    ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=4, fontsize='small')
    fig.show()
    return fig


def plota_budget_Zll(bud, limsx=(1e-4,150), limsy=(1,5e3)):
    bud2 = bud.copy()
    fig = mplt.figure(figsize=(12, 10))
#     fig = mplt.figure(figsize=(9,6))

    gs = mplt.GridSpec(2, 1)
    gs.update(left=0.15, right=0.98, top=0.96, bottom=0.22, hspace=0.08)

    ax1 = mplt.subplot(gs[0,0])
    ax2 = mplt.subplot(gs[1,0],sharex=ax1)
    
    ax1.grid(True)
    ax2.grid(True)
    
    N = len(bud)
    for i in range(len(bud2)):
        Z = bud2[-1].Zll
        if Z is None or len(Z)>0:
            ind = bud2.ang_freq >=0
            freq = bud2.ang_freq[ind]/2/np.pi * 1e-9
            Z = bud2.Zll[ind] * elbud_mod._FACTOR['Zll']
            lab = bud2[-1].name
            cor = i/N if i % 2 else (N-i)/N
            ax1.fill_between(freq, Z.real, color=mplt.cm.jet(cor), label=lab)
            ax2.fill_between(freq, Z.imag, color=mplt.cm.jet(cor), label=lab)
            
            lw = 0.5
            lc = [1,1,1]
            lc = [0,0,0]
            ax1.plot(freq, Z.real, color=lc,linewidth=lw)
            ax2.plot(freq, Z.imag, color=lc,linewidth=lw)
        bud2.pop()

    ax1.set_ylabel(r'Re'+imp.Budget._YLABEL['Zll'])
    ax2.set_ylabel(r'Im'+imp.Budget._YLABEL['Zll'])
    ax2.set_xlabel(r'Frequency [GHz]')

    mplt.setp(ax1.get_xticklabels(), visible=False)
    
    mix,mx = limsx
    ax1.set_xlim([mix,mx])

    miy,may = limsy
    ax1.set_ylim([miy,may])
    ax2.set_ylim([-may,-miy])

    ax2.legend(loc='upper center', ncol=4, bbox_to_anchor=(0.5, -0.22), fontsize='small')
    return fig

In [13]:
def create_budget(name):
   
    elems = {
        # Thiago informed the number of bellow and blw_bpm_blw by email (2017/05/08)
        'comp_bellows': 
            {'betax': 6.6, 'betay': 8.8, 'quantity': 20},
        'comp_blw_bpm_blw': 
            {'betax': 6.6, 'betay': 8.8, 'quantity': 120},
        'comp_dcct': 
            {'betax': 2.5, 'betay': 22.0, 'quantity': 2},  # 13C4 and 14C4
        'comp_long_kicker': 
            {'betax': 5.0, 'betay': 5.0, 'quantity': 1},
        'comp_pump_slots': 
            {'betax': 16.0, 'betay': 7.0, 'quantity': 100}, # I assumed 5 per arch
        # Thiago informed the number of radiation masks by email (2017/05/08)
        'comp_rad_mask': 
            {'betax': 6.6, 'betay': 11.0, 'quantity': 360},
        'comp_scraper_h':
            {'betax': 16.0, 'betay': 7.0, 'quantity': 1},  # 01SA
        'comp_scraper_v':
            {'betax': 16.0, 'betay': 7.0, 'quantity': 1},  # 01SA
        'comp_sl_gsl07': 
            {'betax': 7.0, 'betay': 7.0, 'quantity': 1},  # 19SP
        'comp_sl_gsl15': 
            {'betax': 7.0, 'betay': 7.0, 'quantity': 1},  # 20SB
        'comp_sl_kicker_h': 
            {'betax': 18.2, 'betay': 7.3, 'quantity': 1},  # 17SA
        'comp_sl_kicker_v': 
            {'betax': 2.5, 'betay': 22.0, 'quantity': 1},  # 16C4
        'comp_sl_monit': 
            {'betax': (7.0+2.5)/2, 'betay': (7.0+22)/2, 'quantity': 2},  # 18SB and 17C4
        'comp_sl_shaker_h': 
            {'betax': 18.2, 'betay': 7.3, 'quantity': 1},  # 01SA
        'comp_sl_shaker_v': 
            {'betax': 2.5, 'betay': 22.0, 'quantity': 1},  # 18C4
        # Thiago informed the number of valve_blocks by email (2017/05/08)
        'comp_valve_block': 
            {'betax': 6.6, 'betay': 11.0, 'quantity': 40},
        'csr': 
            {'betax': 2.0, 'betay': 25.0, 'quantity': 1},
        'rw_apu22': 
            {'betax': (16 + 3*1.6)/4, 'betay': (16 + 3*1.6)/4, 'quantity': 4},
        'rw_apu58': 
            {'betax': 1.6, 'betay': 1.6, 'quantity': 1},
        'rw_bc': 
            {'betax': 0.4, 'betay':   5.2, 'quantity': 20},
        'rw_delta22': 
            {'betax': 1.9, 'betay': 1.9, 'quantity': 1},
        'rw_delta52': 
            {'betax': 1.6, 'betay': 1.6, 'quantity': 1},
        'rw_dipk': 
            {'betax': 18.0, 'betay': 6.7, 'quantity': 1},
        # For fast_corr: bx = (10*3+7*2+1*2+12*1)/8    by = (7*3+4*2+5*2+13*1)/8
        'rw_fast_corr': 
            {'betax': 7.2, 'betay': 6.5, 'quantity': 80},
        'rw_nlk': 
            {'betax': 18.2, 'betay': 7.3, 'quantity': 1},
        'rw_pipe': 
            {'betax': 6.0, 'betay': 11.0, 'quantity': 1},
        # The chamber for the APU22 and APU58 are equal. We have one of them
        # at a high beta section and four at low beta. The betas used here
        # were calculated at the end of the undulador:
        'trans_apu': 
            {'betax': (16 + 4*1.8)/5, 'betay': (3.7 + 4*1.8)/5, 'quantity': 5},
        'trans_bc': 
            {'betax': 0.4, 'betay': 5.2, 'quantity': 20},
        # The Delta22 chamber is longer than the Delta52's.
        'trans_delta22': 
            {'betax': 2.6, 'betay': 2.6, 'quantity': 1},
        'trans_delta52': 
            {'betax': 1.8, 'betay': 1.8, 'quantity': 1},
        # The cavity considered here is the Super conducting cavity:
        'trans_rf_cav': 
            {'betax': 7.3, 'betay': 7.3, 'quantity': 1},
        'trans_nlk_and_dipk': 
            {'betax': 18.2, 'betay': 7.1, 'quantity': 1},
        }
        
    bud = imp.Budget(name=name)
    for ele, vals in elems.items():
        if vals['quantity'] <= 0:
            continue
        
        folder = os.path.join('..','elements', ele)
        fil = os.listdir(folder)
        fil = [f for f in fil if f.endswith('.pickle')]
        if len(fil) > 1:
            raise ValueError('To many files for component: ' + ele)
        if not fil:
            raise ValueError('Data not found for component: ' + ele)
        
        el = imp.load_element(os.path.join(folder, fil[0]))
        for k, v in vals.items():
            setattr(el, k, v)
        bud.append(el)

    return bud

def group_budget(bud, groups):
    bud_res = imp.Budget(name=bud.name)
    for name, vals in groups.items():
        bud2 = imp.Budget()
        for i, el in enumerate(bud):
            if el.name in vals:
                bud2.append(el)
        el2 = bud2.budget2element()
        el2.name = name
        bud_res.append(el2)
    
    names = reduce(opr.or_, groups.values())
    for el in bud:
        if el.name not in names:
            bud_res.append(el)
    return bud_res
        

# Define ring Model

In [4]:
mod = si_lat.create_accelerator()
idx = pyaccel.lattice.find_indices(mod, 'frequency', 0, comparison=lambda x, y: x>y)[0]
mod[idx].voltage = 1.75e6

In [5]:
twiss, *_ = pyaccel.optics.calc_twiss(mod)
eqpar = pyaccel.optics.EqParamsFromBeamEnvelope(mod)

In [6]:
print(eqpar)


Energy [GeV]                    : 3
Energy Deviation [%]            : 0
J1, J2, J3                      : 1.299, 1, 1.701
tau1, tau2, tau3 [ms]           : 16.81, 21.85, 12.85
alpha1, alpha2, alpha3 [Hz]     : 59.48, 45.77, 77.84
tune1, tune2, tune3 [Hz]        : 0.09599, 0.1518, 0.003554
momentum compaction x 1e4       : 1.636
energy loss [keV]               : 474.9
overvoltage                     : 3.685
sync phase [°]                  : 164.3
mode 1 emittance [nm.rad]       : 0.2493
mode 2 emittance [pm.rad]       : 2.027e-26
natural espread [%]             : 0.08511
bunch length [mm]               : 3.232
RF energy accep. [%]            : 4.006


In [7]:
ring = si.create_ring()
si.update_from_pymodels(ring, mod)
print(ring)

Lattice Version             :   SI.v25.01-s05.02  
Circumference [m]           :       518.387       
Revolution Period [us]      :        1.729        
Revolution Frequency [kHz]  :       578.318       
Energy [GeV]                :        3.000        
U0 [keV]                    :       474.890       
Momentum Compaction         :       1.64e-04      
Harmonic Number             :         864         
Current [mA]                :       100.000       
Current per Bunch [mA]      :        0.116        
Synchrotron Tune            :       0.00355       
Tunes x/y                   :    49.096/14.152    
Chromaticities x/y          :     2.500/2.500     
Damping Times x/y/e [ms]    :   16.8/ 21.8 /12.8  
Energy Spread [%]           :        0.0851       
Bunch Length [mm]           :        3.232        



## Beam Spectrum

In [10]:
wp = np.linspace(0, 60e9, 1000) * 2*np.pi
max_azi = 1
spec = ring.calc_spectrum(wp, ring.bunlen, max_rad=1, max_azi=max_azi)

In [11]:
fig, ax = plt.subplots(1, 1, figsize=(9, 5))
ax.grid(True, alpha=0.5, ls='--')
ax.set_xlabel('Frequency [GHz]')
ax.set_ylabel('Beam Spectrum')
freq = wp/2/np.pi/1e9
for k, v in spec.items():
    spc = spec[k]**2
    ax.plot(freq, spc, label=r'$h_{{{0:d}{1:d}}}$'.format(*k))

ax.set_yscale('log')
ax.set_ylim([1e-4, 1.1])
ax.set_xlim([freq.min(), freq.max()])
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=max_azi+1, fontsize='xx-small')
fig.tight_layout()

# Create and save Budget

In [10]:
fig, _ = pyaccel.graphics.plot_twiss(mod, symmetry=20, grid=True, twiss=twiss)

In [11]:
betax = np.trapz(twiss.betax,x=twiss.spos)/twiss.spos[-1]
betay = np.trapz(twiss.betay,x=twiss.spos)/twiss.spos[-1]
print("Average Betax = {0:7.2f}          Average Beta = {1:7.2f}".format(betax,betay))

Average Betax =    5.85          Average Beta =   11.16


In [12]:
bpm_idx = pyaccel.lattice.find_indices(mod, 'fam_name','BPM')
bpm_bx  = np.sum(twiss.betax[bpm_idx])/len(bpm_idx)
bpm_by  = np.sum(twiss.betay[bpm_idx])/len(bpm_idx)
print("Average Betax @ BPMs = {0:7.2f}          Average Betay @ BPMs= {1:7.2f}".format(bpm_bx,bpm_by))

Average Betax @ BPMs =    6.65          Average Betay @ BPMs=    9.38


In [14]:
bud = create_budget(name='official')
# bud.save(overwrite=True)

In [15]:
print(bud)

                    official                    
    Element    :  Quantity    Betax      Betay   
Bellows        :     20        6.6        8.8    
Blw BPM Blw    :    120        6.6        8.8    
DCCT           :     2         2.5        22.0   
Long Kicker    :     1         5.0        5.0    
Pump Slots     :    100        16.0       7.0    
Rad Mask       :    360        6.6        11.0   
Scraper H      :     1         16.0       7.0    
Scraper V      :     1         16.0       7.0    
SL GSL07       :     1         7.0        7.0    
SL GSL15       :     1         7.0        7.0    
SL KICKER H    :     1         18.2       7.3    
SL KICKER V    :     1         2.5        22.0   
SL MONIT       :     2         4.8        14.5   
SL SHAKER H    :     1         18.2       7.3    
SL SHAKER V    :     1         2.5        22.0   
Valve Block    :     40        6.6        11.0   
CSR            :     1         2.0        25.0   
RW APU22       :     4         5.2        5.2    
R

## Group Budget

In [18]:
groups = {
    'Trans': {'Trans APU', 'Trans BC', 'Trans Delta22', 'Trans Delta52', 'Trans NLK and DipK'},
    'ResWall': {
        'RW APU22', 'RW APU58', 'RW BC', 'RW Delta22', 'RW Delta52',
        'RW DipK', 'RW Fast Corr', 'RW NLK', 'RW Pipe'},
    'Striplines': {
        'SL GSL07', 'SL GSL15', 'SL KICKER H', 'SL KICKER V', 'SL MONIT',
        'SL SHAKER H', 'SL SHAKER V'},
    'Scrapers': {'Scraper H', 'Scraper V'},
    }
bud2 = group_budget(bud, groups)
bud2.name = 'official_grouped'

In [19]:
print(bud2)

                official_grouped                
    Element    :  Quantity    Betax      Betay   
Trans          :     1         1.0        1.0    
ResWall        :     1         1.0        1.0    
Striplines     :     1         1.0        1.0    
Scrapers       :     1         1.0        1.0    
Bellows        :     20        6.6        8.8    
Blw BPM Blw    :    120        6.6        8.8    
DCCT           :     2         2.5        22.0   
Long Kicker    :     1         5.0        5.0    
Pump Slots     :    100        16.0       7.0    
Rad Mask       :    360        6.6        11.0   
Valve Block    :     40        6.6        11.0   
CSR            :     1         2.0        25.0   
Trans RF CAV   :     1         7.3        7.3    




In [20]:
bud2.save(overwrite=True)

# Load and Analyse Budget

In [112]:
bud = imp.load_budget('official.pickle')
# bud = imp.load_budget('official_grouped.pickle')
bud.max_ang_freq = 100e9 * 2*np.pi


In [115]:
mplt.figure(figsize=(7, 4))
idx = bud.ang_freq > 0
mplt.plot(bud.ang_freq[idx]/2/np.pi/1e9, bud.Zll.real[idx]/1e3, label='$\Re(Z_\parallel)$')
mplt.plot(bud.ang_freq[idx]/2/np.pi/1e9, bud.Zll.imag[idx]/1e3, label='$\Im(Z_\parallel)$')
# mplt.yscale('log')
mplt.xlabel('$f$ [GHz]')
mplt.ylabel('$Z_\parallel$ [$k\Omega$]')
# mplt.xlim([0, 50])
mplt.legend(fontsize=12)
mplt.tight_layout()
mplt.show()

In [94]:
mplt.figure(figsize=(12, 3))
mplt.plot(bud.pos, bud.Wll/1e12)
mplt.yscale('log')
mplt.ylim([1e-9, 1e7])
mplt.tight_layout()
mplt.show()

## Make some figures

In [22]:
fig = plota_budget_Zll(bud, limsx=(1e-4, 100), limsy=(1e-3, 10))
fig.show()
# fig.savefig('impedance_longitudinal.png')

183869 183869
183869 183869
183869 183869
183614 183614
183614 183614
182342 182342
182088 182088
182088 182088
181688 181688
181288 181288
180888 180888
180488 180488
180088 180088
179688 179688
179688 179688
179287 179287
174287 174287
173271 173271
173271 173271
146507 146507
137588 137588
137588 137588
101552 101552
92606 92606
85161 85161
84841 84841
80291 80291
64713 64713
14652 14652
10464 10464


In [46]:
for freq, Zl in zip(total_freq, total_Zl):
    print(freq.size, Zl.size)

183869 1273
183869 10000
178869 2033
177853 641
177533 9101
172983 31157
157405 100123
107344 8377
103156 20927
92693 176221
4583 5602
1782 3563


In [38]:
max_pts = []
for Zl in total_Zl:
    max_pts.append(Zl.size)
    
idx_max = np.argmax(max_pts)

new_Zl = []
for idx, Zl in enumerate(total_Zl):
    new_Zl.append(np.interp(total_freq[idx_max], total_freq[idx], total_Zl[idx]))

ValueError: fp and xp are not of the same length.

In [37]:
np.interp?

In [23]:
fig = plota_budget_Zt(bud, pl='Zdy', limsx=(1e-4, 80), limsy=(1e-3, 3))
fig.show()
fig.savefig('impedance_dipolar_vertical.png')

In [24]:
fig = plota_budget_Zt(bud, pl='Zdx', limsx=(1e-3, 150), limsy=(1e-3, 3))
fig.show()
fig.savefig('impedance_dipolar_horizontal.png')

In [25]:
fig = plota_budget_Zt(bud, pl='Zqx', limsx=(1e-3, 150), limsy=(1, 3e2))
fig.show()
fig.savefig('impedance_detuning_horizontal.png')

In [26]:
fig = plota_budget_Zt(bud, pl='Zqy', limsx=(1e-3, 150), limsy=(1, 3e2))
fig.show()
fig.savefig('impedance_detuning_vertical.png')

In [27]:
ring.num_bun = 1
ring.total_current = 1e-3
summ = ring.budget_summary(bud).T
summ

,lsf,zln,pls,kdx,kdy,kqx,kqy,ktx,kty,ndx,ndy,nqx,nqy,ntx,nty
name,KLoss,Zl/n,PLoss,Kdx,Kdy,Kqx,Kqx,Kx,Ky,TuShdx,TuShdy,TuShqx,TuShqy,TuShx,TuShy
unit,[V/pC],[mOhm],[W],[kV/pC],[kV/pC],[kV/pC],[kV/pC],[kV/pC],[kV/pC],1/10^3,1/10^3,1/10^3,1/10^3,1/10^3,1/10^3
latex_unit,[V/pC],[$m\Omega$],[W],[kV/pC],[kV/pC],[kV/pC],[kV/pC],[kV/pC],[kV/pC],$\times10^{-3}$,$\times10^{-3}$,$\times10^{-3}$,$\times10^{-3}$,$\times10^{-3}$,$\times10^{-3}$
latex_name,$\kappa_{Loss}$,$Z_L/n|_{eff}$,$P_{Loss}$,$\beta_x\kappa_x^D$,$\beta_y\kappa_y^D$,$\beta_x\kappa_x^Q$,$\beta_y\kappa_y^Q$,$\beta_x\kappa_x$,$\beta_y\kappa_y$,$\Delta\nu_x^D$,$\Delta\nu_y^D$,$\Delta\nu_x^Q$,$\Delta\nu_y^Q$,$\Delta\nu_x$,$\Delta\nu_y$
Trans,0.079706,-7.649625,0.137824,-0.84013,-4.10826,0.686677,-0.19598,-0.153453,-4.304241,-0.038534,-0.188434,0.031496,-0.008989,-0.007038,-0.197424
ResWall,5.546455,-41.260844,9.59067,-12.869897,-21.754455,3.798617,-3.112121,-9.071279,-24.866575,-0.590306,-0.997816,0.174232,-0.142744,-0.416074,-1.14056
Striplines,2.097915,-0.89385,3.627616,-1.335885,-1.813839,-0.390013,-0.365275,-1.725898,-2.179115,-0.061273,-0.083196,-0.017889,-0.016754,-0.079162,-0.09995
Scrapers,0.476113,-0.351083,0.823272,NaN,NaN,3.780064,-0.204479,3.780064,-0.204479,NaN,NaN,0.173381,-0.009379,0.173381,-0.009379
Bellows,0.789672,-0.695485,1.365463,-0.452843,-0.60601,-0.000452,0.00061,-0.453294,-0.6054,-0.020771,-0.027796,-0.000021,0.000028,-0.020791,-0.027768
Blw BPM Blw,9.977665,-13.143116,17.252909,-6.124873,-8.166498,NaN,NaN,-6.124873,-8.166498,-0.280931,-0.374574,NaN,NaN,-0.280931,-0.374574


In [11]:
print(summ[4:].sum())

lsf    31.811575
zln   -95.183141
pls    55.007080
kdx   -31.201291
kdy   -45.663344
kqx     0.344338
kqy     2.574390
ktx   -30.856953
kty   -43.088954
ndx    -1.431116
ndy    -2.094450
nqx     0.015794
nqy     0.118080
ntx    -1.415322
nty    -1.976370
dtype: float64


In [16]:
fig, (ax, ay) = mplt.subplots(2, 1, figsize=(7, 6), sharex=True)

serie = summ['zln']
lab = serie['latex_name'] + ' ' + serie['latex_unit']
ax = serie[4:].plot(kind='bar', rot=45, fontsize='x-small', ylabel=lab, ax=ax)
ax.grid(True, alpha=0.5, ls='--')


serie = summ['lsf']
lab = serie['latex_name'] + ' ' + serie['latex_unit']
ay = serie[4:].plot(kind='bar', rot=90, fontsize='x-small', ylabel=lab, ax=ay, position=0.5)
ay.grid(True, alpha=0.5, ls='--')

fig.tight_layout()
fig.savefig('effective_z_over_n_and_loss_factor.png')
fig.show()

In [17]:
fig, (ax, ay) = mplt.subplots(2, 1, figsize=(9, 6), sharex=True)

ax.set_title('Tune-Shifts @ 1 mA Single-Bunch')
serie = summ['ndx']
lab = serie['latex_name'] + ' ' + serie['latex_unit']
ax = serie[4:].plot(kind='bar', rot=45, fontsize='x-small', ax=ax, ylabel=lab)
ax.grid(True, alpha=0.5, ls='--')

serie = summ['ndy']
lab = serie['latex_name'] + ' ' + serie['latex_unit']
ay = serie[4:].plot(kind='bar', rot=90, fontsize='x-small', ax=ay, ylabel=lab)
ay.grid(True, alpha=0.5, ls='--')

fig.tight_layout()
fig.savefig('coherent_tune_shifts.png')
fig.show()